In [11]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import folium       

In [12]:
CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
PERIOD = "202510"

In [13]:
file_name = f"JC-{PERIOD}-citibike-tripdata.zip"
url = f"{CITIBIKE_INDEX_URL}/{file_name}"
url

'https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip'

In [14]:
# eval: true
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

file_name = f"JC-{PERIOD}-citibike-tripdata.zip"

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)

zip_path = output_dir / file_name


## Dowloading the Zip file
url = f"{CITIBIKE_INDEX_URL}/{file_name}"

print(f"Downloading: {url}")
urlretrieve(url, zip_path)

print(f"Saved ZIP file to: {zip_path}")


with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)

print(f"Extracted files into: {output_dir}")

zip_path.unlink()
print("ZIP file removed.")

Downloading: https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip
Saved ZIP file to: ../data/citibike/JC-202510-citibike-tripdata.zip
Extracted files into: ../data/citibike
ZIP file removed.


### Read the Data

In [15]:
file_name  = 'JC-202510-citibike-tripdata.csv'
citibike_202510 = pd.read_csv(f'{output_dir}/{file_name}')
citibike_202510.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,929852F0DC04F49C,electric_bike,2025-10-08 06:59:53.767,2025-10-08 07:04:08.012,Warren St,JC006,Essex Light Rail,NaN,40.721124,-74.038051,NaN,NaN,member
1,7E6F8FADF45B1D8E,electric_bike,2025-10-08 08:13:15.096,2025-10-08 08:18:16.168,Newark Ave,JC032,Essex Light Rail,NaN,40.721525,-74.046305,NaN,NaN,member
2,9B679869FCA8F845,electric_bike,2025-10-09 10:10:26.020,2025-10-09 10:20:08.505,Liberty Light Rail,JC052,Newport PATH,NaN,40.711242,-74.055701,NaN,NaN,member
3,E2051DD457BC4076,electric_bike,2025-10-09 11:34:22.937,2025-10-09 11:48:49.501,Hoboken Ave at Monmouth St,JC105,Newport PATH,NaN,40.735208,-74.046964,NaN,NaN,member
4,239AC07371E414EC,electric_bike,2025-10-23 19:24:16.237,2025-10-23 19:28:22.592,Union St & Bergen Ave,JC122,Garfield Light Rail,JC119,40.715750,-74.078870,40.710526,-74.070112,member


### Downloading year 2025

In [16]:
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    
    """
    YEAR = year
    MONTH =  [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]

    periods = []

    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
    # print(periods)
    return periods

periods  = period_iterator(year = ['2025'],start_m = 1, stop_m = 12)

periods

['202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [19]:
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)


for i in periods:

    try:
        file_name = f"JC-{i}-citibike-tripdata.csv.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    except (HTTPError, URLError, FileNotFoundError):
        file_name = f"JC-{i}-citibike-tripdata.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)
    print(f'{file_name}  Extracted')
    zip_path.unlink()
    print(f"{file_name} removed.")

JC-202502-citibike-tripdata.csv.zip  Extracted
JC-202502-citibike-tripdata.csv.zip removed.
JC-202503-citibike-tripdata.csv.zip  Extracted
JC-202503-citibike-tripdata.csv.zip removed.
JC-202504-citibike-tripdata.csv.zip  Extracted
JC-202504-citibike-tripdata.csv.zip removed.
JC-202505-citibike-tripdata.csv.zip  Extracted
JC-202505-citibike-tripdata.csv.zip removed.
JC-202506-citibike-tripdata.csv.zip  Extracted
JC-202506-citibike-tripdata.csv.zip removed.
JC-202507-citibike-tripdata.csv.zip  Extracted
JC-202507-citibike-tripdata.csv.zip removed.
JC-202508-citibike-tripdata.csv.zip  Extracted
JC-202508-citibike-tripdata.csv.zip removed.
JC-202509-citibike-tripdata.csv.zip  Extracted
JC-202509-citibike-tripdata.csv.zip removed.
JC-202510-citibike-tripdata.zip  Extracted
JC-202510-citibike-tripdata.zip removed.
JC-202511-citibike-tripdata.csv.zip  Extracted
JC-202511-citibike-tripdata.csv.zip removed.
JC-202512-citibike-tripdata.csv.zip  Extracted
JC-202512-citibike-tripdata.csv.zip remov

In [20]:
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    year list of strings
    """
    YEAR = year
    MONTH =  [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]

    periods = []

    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
    # print(periods)
    return periods

PERIODS = period_iterator(["2025"],0,12)
PERIODS

['202501',
 '202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [21]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import requests

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

for i in PERIODS:
    urls = [
    f"{CITIBIKE_INDEX_URL}/JC-{i}-citibike-tripdata.csv.zip",
    f"{CITIBIKE_INDEX_URL}/JC-{i}-citibike-tripdata.zip",
]

    last_error = None

    for url in urls:
        try:
            print(f"Trying: {url}")

            response = requests.get(url, timeout=30)
            response.raise_for_status()

            with ZipFile(BytesIO(response.content), "r") as zip_ref:
                zip_ref.extractall(output_dir)

            print(f"Success: {url}")
            break

        except Exception as e:
            print(f"Failed: {url}")
            last_error = e

    else:
        raise RuntimeError(f"Both URLs failed for period {i}") from last_error

Trying: https://s3.amazonaws.com/tripdata/JC-202501-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202501-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202502-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202502-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202503-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202503-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202504-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202504-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202505-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202505-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202506-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202506-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202507-citibi

### Removing __MACOSX

In [22]:
import shutil

shutil.rmtree(output_dir / "__MACOSX")